In [0]:
%pip install opencv-python-headless==4.8.1.78 "numpy<2.0"

In [0]:
%restart_python

In [0]:
import os
import cv2
import numpy as np
from pathlib import Path
import pandas as pd
from config import DATASET_DIR, TISSUE_THRESHOLD
import mlflow

In [0]:
mlflow.set_experiment("/Workspace/Users/shrestharoyce24@gmail.com/Databricks MLOps and MLFlow with Computer Vision Models/ovarian_cancer_pipeline")

In [0]:
# Can be run if the dataset includes white glass borders, out-of-focus blurs, or black pen marks
def is_valid_tissue(image_path: str, min_tissue_ratio: float) -> bool:

    bgr_img = cv2.imread(image_path)
    if bgr_img is None:
        return False

    hsv_img = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2HSV)
    saturation = hsv_img[:,:,1]

    _, tissue_mask = cv2.threshold(saturation, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    tissue_ratio = np.count_nonzero(tissue_mask) / tissue_mask.size

    return tissue_ratio >= min_tissue_ratio

valid_file_paths = []
full_paths = []
full_labels = []
valid_labels = []

class_names = sorted([d for d in os.listdir(DATASET_DIR) if not d.startswith(".")])
# print(class_names)

for class_idx, class_name in enumerate(class_names):
    class_folder = os.path.join(DATASET_DIR, class_name)
    if not os.path.isdir(class_folder):
        continue

    for fname in os.listdir(class_folder):
        if fname.lower().endswith(".jpg"):
            fpath = os.path.join(class_folder, fname)
            full_paths.append(fpath)
            full_labels.append(class_idx)
            '''if is_valid_tissue(fpath, min_tissue_ratio = TISSUE_THRESHOLD):
                valid_file_paths.append(fpath)
                valid_labels.append(class_idx)'''

valid_file_paths = full_paths

In [0]:
total_count = len(full_paths)
valid_count = len(valid_file_paths)
attrition_count = total_count - valid_count
attrition_rate = (attrition_count / total_count ) if total_count > 0 else 0.0

In [0]:
with mlflow.start_run(run_name = 'ovarian_pipeline_run') as run:
    run_id = run.info.run_id

    mlflow.log_param("tissue_threshold", TISSUE_THRESHOLD)
    mlflow.log_metrics({
        "total_images_count": total_count,
        "valid_images_count": valid_count,
        "images_discraded": attrition_count,
        "attrition_rate": attrition_rate
    })

In [0]:
df = pd.DataFrame({
    "image_path": full_paths,
    "label": full_labels
})

spark_df = spark.createDataFrame(df)
spark_df.write.format("delta").mode("overwrite").saveAsTable("CompVis_OD.data.image_paths")